# Framework Fundamentals

**Module:** 09-llm-frameworks

**Notebook:** `01-framework-fundamentals.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Why Frameworks?** with clear contracts and failure modes
- Explain and apply **Core Abstractions** with clear contracts and failure modes
- Explain and apply **Framework vs Raw SDK** with clear contracts and failure modes
- Explain and apply **Common Pitfalls** with clear contracts and failure modes
- Explain and apply **Evaluation Mindset** with clear contracts and failure modes
- Explain and apply **Minimal Composition Pattern** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Framework Fundamentals

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Why Frameworks?**
2. **Core Abstractions**
3. **Framework vs Raw SDK**
4. **Common Pitfalls**
5. **Evaluation Mindset**
6. **Minimal Composition Pattern**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Why Frameworks?

### Definition
**Why Frameworks?** is a core building block in 01-framework-fundamentals within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Why Frameworks? typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Why Frameworks?: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Why Frameworks? as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Why Frameworks? as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Why Frameworks?
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Why Frameworks? when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Why Frameworks? improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Why Frameworks?" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Why Frameworks?"
    notebook: str = "01-framework-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
# Framework mental model: compose runnable steps
class Runnable:
    def __init__(self, fn): self.fn = fn
    def __or__(self, other):
        return Runnable(lambda x: other.fn(self.fn(x)))
    def invoke(self, x): return self.fn(x)

prompt = Runnable(lambda q: f"Answer briefly: {q}")
model = Runnable(lambda p: f"<model-output for: {p[:40]}...>")
parse = Runnable(lambda t: {"text": t, "n": len(t)})
chain = prompt | model | parse
print(chain.invoke("What is LCEL?"))


In [ ]:
# Index → retrieve → generate sketch (RAG-shaped)
NODES = [{"id": 1, "text": "LCEL composes runnables with |"}, {"id": 2, "text": "LlamaIndex focuses on data indexes"}]

def retrieve(q, k=1):
    return sorted(NODES, key=lambda n: len(set(q.lower().split()) & set(n["text"].lower().split())), reverse=True)[:k]

def generate(q, nodes):
    ctx = " | ".join(n["text"] for n in nodes)
    return f"Q: {q}\nCTX: {ctx}\nA: Based on context, {nodes[0]['text']}."

print(generate("What is LCEL?", retrieve("LCEL compose")))


## Core Abstractions

### Definition
**Core Abstractions** is a core building block in 01-framework-fundamentals within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Core Abstractions typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Core Abstractions: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Core Abstractions as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Core Abstractions as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Core Abstractions
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Core Abstractions when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Core Abstractions" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Core Abstractions"
    notebook: str = "01-framework-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


In [ ]:
# Demo: decision table for applying "Core Abstractions"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_core_abstrac", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


### Worked scenario — Core Abstractions

**Situation:** A team wants to productionize a feature involving **Core Abstractions**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Framework vs Raw SDK

### Definition
**Framework vs Raw SDK** helps you choose among alternatives using explicit criteria rather than hype.

### Why it matters
In LLM frameworks, weak designs around Framework vs Raw SDK typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
List options, define criteria (quality, cost, latency, ops, lock-in), score with evidence, document the decision.

### Intuition
Explain Framework vs Raw SDK as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Framework vs Raw SDK as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Framework vs Raw SDK
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Framework vs Raw SDK when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Framework vs Raw SDK" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Framework vs Raw SDK"
    notebook: str = "01-framework-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# Framework mental model: compose runnable steps
class Runnable:
    def __init__(self, fn): self.fn = fn
    def __or__(self, other):
        return Runnable(lambda x: other.fn(self.fn(x)))
    def invoke(self, x): return self.fn(x)

prompt = Runnable(lambda q: f"Answer briefly: {q}")
model = Runnable(lambda p: f"<model-output for: {p[:40]}...>")
parse = Runnable(lambda t: {"text": t, "n": len(t)})
chain = prompt | model | parse
print(chain.invoke("What is LCEL?"))


In [ ]:
# Index → retrieve → generate sketch (RAG-shaped)
NODES = [{"id": 1, "text": "LCEL composes runnables with |"}, {"id": 2, "text": "LlamaIndex focuses on data indexes"}]

def retrieve(q, k=1):
    return sorted(NODES, key=lambda n: len(set(q.lower().split()) & set(n["text"].lower().split())), reverse=True)[:k]

def generate(q, nodes):
    ctx = " | ".join(n["text"] for n in nodes)
    return f"Q: {q}\nCTX: {ctx}\nA: Based on context, {nodes[0]['text']}."

print(generate("What is LCEL?", retrieve("LCEL compose")))


## Common Pitfalls

### Definition
**Common Pitfalls** is a core building block in 01-framework-fundamentals within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Common Pitfalls typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Common Pitfalls: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Common Pitfalls as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Common Pitfalls as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Common Pitfalls
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Common Pitfalls when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Common Pitfalls" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Common Pitfalls"
    notebook: str = "01-framework-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Common Pitfalls"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Common Pitfalls"}
strong = {"definition": "Common Pitfalls", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Common Pitfalls"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Common Pitfalls", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Common Pitfalls

**Situation:** A team wants to productionize a feature involving **Common Pitfalls**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Evaluation Mindset

### Definition
**Evaluation Mindset** is a core building block in 01-framework-fundamentals within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Evaluation Mindset typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Evaluation Mindset: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Evaluation Mindset as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Evaluation Mindset as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Evaluation Mindset
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Evaluation Mindset when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Evaluation Mindset" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Evaluation Mindset"
    notebook: str = "01-framework-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
GOLDEN = [{"x": "charged twice", "y": "billing"}, {"x": "SSO", "y": "auth"}]

def predict(x: str) -> str:
    return "billing" if "charge" in x.lower() or "invoice" in x.lower() else ("auth" if "sso" in x.lower() or "login" in x.lower() else "other")

def eval_prompt(name: str):
    rows = [(g["x"], g["y"], predict(g["x"])) for g in GOLDEN]
    acc = sum(y == p for _, y, p in rows) / len(rows)
    return {"name": name, "accuracy": acc, "rows": rows}

print(eval_prompt("v1"))


In [ ]:
import hashlib

def prompt_version(text: str) -> str:
    return "pv_" + hashlib.sha1(text.encode()).hexdigest()[:10]

a = "Label tickets carefully"
b = "Label tickets as billing|auth|outage|other. JSON only."
print(prompt_version(a), prompt_version(b))


## Minimal Composition Pattern

### Definition
**Minimal Composition Pattern** is a core building block in 01-framework-fundamentals within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Minimal Composition Pattern typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Minimal Composition Pattern: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Minimal Composition Pattern as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Minimal Composition Pattern as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Minimal Composition Pattern
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Minimal Composition Pattern when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Minimal Composition Pattern" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Minimal Composition Pattern"
    notebook: str = "01-framework-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Minimal Composition Pattern"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Minimal Composition Pattern"}
strong = {"definition": "Minimal Composition Pattern", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Minimal Composition Pattern"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Minimal Composition Pattern", "passed": len(checks)-len(failed), "failed": failed})


In [ ]:
# Demo: decision table for applying "Minimal Composition Pattern"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_minimal_comp", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


### Worked scenario — Minimal Composition Pattern

**Situation:** A team wants to productionize a feature involving **Minimal Composition Pattern**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **Framework Fundamentals**.

| Topic | Do | Don't |
|-------|----|-------|
| Why Frameworks? | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Core Abstractions | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Framework vs Raw SDK | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Common Pitfalls | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Evaluation Mindset | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Minimal Composition Pattern | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Why Frameworks? | Key concept covered in this notebook; see its section for definition and pitfalls |
| Core Abstractions | Key concept covered in this notebook; see its section for definition and pitfalls |
| Framework vs Raw SDK | Key concept covered in this notebook; see its section for definition and pitfalls |
| Common Pitfalls | Key concept covered in this notebook; see its section for definition and pitfalls |
| Evaluation Mindset | Key concept covered in this notebook; see its section for definition and pitfalls |
| Minimal Composition Pattern | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Framework Fundamentals** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **09-llm-frameworks**.


## Try It Yourself

1. Implement a failing test/fixture for **Why Frameworks?**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Core Abstractions**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Framework vs Raw SDK**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Common Pitfalls**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Evaluation Mindset**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
